# Part 2 – RFM Segmentation & Retention Strategy

**Snapshot date:** 2025-09-30  
**RFM window:** 180 days before snapshot (orders placed 2025-04-03 → 2025-09-30)  
**Non-RFM signals added:** support tickets (90d), web/app engagement (30d), campaign history, return rate


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

DATA = Path('data')
if not DATA.exists() or not any(DATA.glob('*.csv')):
    DATA = Path('../../dataset')

CHARTS = Path('outputs/charts')
CHARTS.mkdir(parents=True, exist_ok=True)
print('Data path:', DATA.resolve())

## 1. Load Data

In [ ]:
# Use the pre-built snapshot — all features are already pre-snapshot safe
snap = pd.read_csv(DATA / 'rfm_modeling_snapshot.csv')
snap['loyalty_tier'] = snap['loyalty_tier'].fillna('None')

interventions = pd.read_csv(DATA / 'intervention_history.csv')
snap = snap.merge(
    interventions[['customer_id', 'last_campaign_received', 'manual_priority_bucket']],
    on='customer_id', how='left'
)

print(f'Rows: {len(snap)}  Columns: {snap.shape[1]}')
print(f'Overall churn rate: {snap["churn_next_60d"].mean():.2%}')
snap[['recency_days','frequency_180d','monetary_180d','return_rate_180d',
      'ticket_count_90d','sessions_30d','avg_discount_pct_180d']].describe()

## 2. RFM Feature Creation

In [ ]:
# Quintile scoring: Recency lower = better → score 5; Frequency/Monetary higher = better → score 5
snap['R'] = pd.qcut(snap['recency_days'], q=5,
                    labels=[5, 4, 3, 2, 1], duplicates='drop').astype(int)
snap['F'] = pd.qcut(snap['frequency_180d'].rank(method='first'), q=5,
                    labels=[1, 2, 3, 4, 5], duplicates='drop').astype(int)
snap['M'] = pd.qcut(snap['monetary_180d'].rank(method='first'), q=5,
                    labels=[1, 2, 3, 4, 5], duplicates='drop').astype(int)
snap['rfm_score'] = snap['R'] + snap['F'] + snap['M']

print('RFM score distribution:')
print(snap['rfm_score'].describe())
print('\nScore range: min', snap['rfm_score'].min(), '  max', snap['rfm_score'].max())

## 3. Additional Non-RFM Signals

In [ ]:
# Composite engagement score (web/app signal)
snap['engagement_score'] = (
    snap['sessions_30d'] +
    snap['campaign_clicks_30d'] +
    snap['email_opens_30d']
)

# Support risk composite (ticket count weighted by negativity)
snap['support_risk'] = snap['ticket_count_90d'] * (1 + snap['negative_ticket_rate_90d'])

print('Signal summaries:')
snap[['engagement_score', 'support_risk', 'return_rate_180d', 'avg_discount_pct_180d']].describe()

## 4. Segmentation Logic

In [ ]:
def assign_segment(row):
    rfm    = row['rfm_score']
    R, M   = row['R'], row['M']
    sup    = row['support_risk']
    engage = row['engagement_score']
    ret    = row['return_rate_180d']
    disc   = row['avg_discount_pct_180d']
    rec    = row['recency_days']
    tenure = row['days_since_signup']
    freq   = row['frequency_180d']

    if rfm >= 13 and sup < 1:
        return 'Champions'
    elif rfm >= 10 and rec > 60:
        return 'Loyal_Slipping'
    elif M >= 4 and (sup >= 2 or ret > 0.35):
        return 'High_Value_Unhappy'
    elif disc >= 0.35 and freq >= 3:
        return 'Discount_Sensitive'
    elif rec > 120 and engage <= 2:
        return 'Dormant'
    elif tenure <= 90:
        return 'New_Customers'
    else:
        return 'Mid_Tier_At_Risk'

snap['segment_name'] = snap.apply(assign_segment, axis=1)

seg_summary = snap.groupby('segment_name').agg(
    count=('customer_id', 'count'),
    churn_rate=('churn_next_60d', 'mean'),
    avg_recency=('recency_days', 'mean'),
    avg_spend=('monetary_180d', 'mean'),
    avg_tickets=('ticket_count_90d', 'mean'),
    avg_sessions=('sessions_30d', 'mean')
).sort_values('churn_rate', ascending=False)

print(seg_summary.to_string())

## 5. Charts

In [ ]:
# Chart 1: Segment size and churn rate side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_summary['count'].sort_values().plot(kind='barh', ax=axes[0], color='#4C9BE8')
axes[0].set_title('Customers per Segment', fontweight='bold')
axes[0].set_xlabel('Count')

seg_summary['churn_rate'].sort_values().plot(kind='barh', ax=axes[1], color='#E85555')
axes[1].set_title('Churn Rate per Segment', fontweight='bold')
axes[1].set_xlabel('Churn Rate')
for i, v in enumerate(seg_summary['churn_rate'].sort_values().values):
    axes[1].text(v + 0.005, i, f'{v:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(CHARTS / '01_segment_overview.png', dpi=120)
plt.show()

In [ ]:
# Chart 2: RFM score distribution
fig, ax = plt.subplots(figsize=(7, 4))
snap['rfm_score'].plot(kind='hist', bins=12, ax=ax, color='#4C9BE8', edgecolor='white')
ax.set_title('RFM Score Distribution (3–15)', fontweight='bold')
ax.set_xlabel('RFM Score'); ax.set_ylabel('Customers')
plt.tight_layout()
plt.savefig(CHARTS / '02_rfm_score_distribution.png', dpi=120)
plt.show()

In [ ]:
# Chart 3: Avg spend vs avg recency by segment (bubble = churn rate)
fig, ax = plt.subplots(figsize=(9, 5))
for seg, row in seg_summary.iterrows():
    ax.scatter(row['avg_recency'], row['avg_spend'],
               s=row['churn_rate'] * 3000, alpha=0.6,
               label=f"{seg} ({row['churn_rate']:.0%})")
    ax.annotate(seg, (row['avg_recency'], row['avg_spend']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xlabel('Avg Recency (days)'); ax.set_ylabel('Avg Spend INR (180d)')
ax.set_title('Segment Map: Recency vs Spend (bubble = churn rate)', fontweight='bold')
plt.tight_layout()
plt.savefig(CHARTS / '03_segment_map.png', dpi=120)
plt.show()

## 6. Save segments.csv

In [ ]:
out_cols = [
    'customer_id', 'segment_name',
    'recency_days', 'frequency_180d', 'monetary_180d',
    'R', 'F', 'M', 'rfm_score',
    'return_rate_180d', 'avg_discount_pct_180d',
    'ticket_count_90d', 'negative_ticket_rate_90d',
    'sessions_30d', 'last_visit_days_ago', 'engagement_score',
    'last_campaign_received', 'manual_priority_bucket',
    'churn_next_60d'
]
snap[out_cols].to_csv('segments.csv', index=False)
print(f'Saved segments.csv — {len(snap)} rows')
snap['segment_name'].value_counts()

## 7. Retention Strategy Summary

Full strategy in `retention_strategy.md`. Key playbook per segment:

| Segment | Action | Priority |
|---|---|:---:|
| Champions | VIP perks, early access, referral rewards | 5 |
| Loyal_Slipping | Personalized re-engagement email + category rec | 2 |
| High_Value_Unhappy | Support resolution call first, then retention credit | 1 |
| Discount_Sensitive | Bundle/threshold offers, avoid flat discounts | 3 |
| Dormant | 3-touch win-back (email→SMS→retargeting), then suppress | 4 |
| New_Customers | Onboarding series + first milestone reward | 6 |
| Mid_Tier_At_Risk | Value-led nudge based on preferred_category | 3 |

## 7b. Save Retention Strategy Markdown

In [ ]:
# Generate and save retention strategy markdown
strategy_lines = ['# Retention Strategy\n']
strategy_lines.append(f'Generated: {pd.Timestamp.now().strftime("%Y-%m-%d")}\n')
strategy_lines.append(f'Snapshot date: 2025-09-30\n')
strategy_lines.append(f'Overall churn rate: {snap["churn_next_60d"].mean():.2%}\n\n')

# Summary table
strategy_lines.append('## Segment Overview\n\n')
strategy_lines.append('| Segment | Count | Churn % | Avg Recency | Avg Spend | Action Priority |\n')
strategy_lines.append('|---|---:|---:|---:|---:|:---:|\n')
for seg, row in seg_summary.iterrows():
    strategy_lines.append(
        f"| {seg} | {row['count']:.0f} | {row['churn_rate']:.1%} | {row['avg_recency']:.0f}d | ₹{row['avg_spend']:.0f} | "
    )
    if seg == 'High_Value_Unhappy':
        strategy_lines.append('**1** |\n')
    elif seg == 'Loyal_Slipping':
        strategy_lines.append('**2** |\n')
    elif seg in ['Discount_Sensitive', 'Mid_Tier_At_Risk']:
        strategy_lines.append('**3** |\n')
    elif seg == 'Dormant':
        strategy_lines.append('**4** |\n')
    elif seg == 'Champions':
        strategy_lines.append('**5** |\n')
    else:
        strategy_lines.append('**6** |\n')

strategy_lines.append('\n## Segment-wise Retention Playbook\n\n')

playbook = {
    'Champions': {
        'desc': 'High RFM, low support issues, brand advocates',
        'action': 'VIP perks, early product access, exclusive community, referral rewards',
        'priority': 5,
        'rationale': 'Maintain loyalty in highest-value segment; low churn but high revenue impact'
    },
    'Loyal_Slipping': {
        'desc': 'Good RFM but recency declining (inactive recently)',
        'action': 'Personalized re-engagement email + product category recommendations based on purchase history',
        'priority': 2,
        'rationale': 'Second highest impact; can reverse trend quickly with right nudge'
    },
    'High_Value_Unhappy': {
        'desc': 'High spend but support issues or high returns',
        'action': 'Proactive support call to resolve issues FIRST, then retention offer (account credit/loyalty points)',
        'priority': 1,
        'rationale': 'URGENT: Highest churn + highest spend = greatest revenue loss risk'
    },
    'Discount_Sensitive': {
        'desc': 'Frequent buyers but heavily discounted, price-conscious',
        'action': 'Bundle offers, tier-based discounts (reward loyalty), avoid flat % discounts',
        'priority': 3,
        'rationale': 'Large segment; must avoid margin erosion while retaining volume'
    },
    'Dormant': {
        'desc': 'Old recency (>120d), low engagement, at high churn risk',
        'action': '3-touch win-back sequence: email (day 0) → SMS (day 5) → retargeting ad (day 10), then suppress',
        'priority': 4,
        'rationale': 'High churn rate; only cost-effective for high-value dormant; suppress low-value after 3 touches'
    },
    'New_Customers': {
        'desc': 'Tenure ≤ 90 days, onboarding phase',
        'action': 'Automated onboarding series, first-purchase follow-up, milestone reward (first repeat, 3rd order)',
        'priority': 6,
        'rationale': 'Lower churn in cohort; focus is habit formation, not retention'
    },
    'Mid_Tier_At_Risk': {
        'desc': 'Moderate RFM, mixed signals, majority of customer base',
        'action': 'Value-led personalized nudge (product recommendations from preferred_category), seasonal campaigns',
        'priority': 3,
        'rationale': 'Largest segment; generic retention efforts; segment by category affinity for better results'
    }
}

for seg in ['High_Value_Unhappy', 'Loyal_Slipping', 'Discount_Sensitive', 'Mid_Tier_At_Risk', 'Dormant', 'Champions', 'New_Customers']:
    if seg in playbook:
        p = playbook[seg]
        strategy_lines.append(f'### {seg}\n\n')
        strategy_lines.append(f'**Description:** {p["desc"]}\n\n')
        strategy_lines.append(f'**Action:** {p["action"]}\n\n')
        strategy_lines.append(f'**Priority:** {p["priority"]} (1=URGENT, 5=maintain, 6=nurture)\n\n')
        strategy_lines.append(f'**Rationale:** {p["rationale"]}\n\n')

strategy_lines.append('\n## Implementation Timeline\n\n')
strategy_lines.append('1. **Week 1:** Launch High_Value_Unhappy support recovery calls\n')
strategy_lines.append('2. **Week 1-2:** Deploy Loyal_Slipping re-engagement email campaign\n')
strategy_lines.append('3. **Week 2:** Design bundle offers for Discount_Sensitive segment\n')
strategy_lines.append('4. **Week 3:** Launch Mid_Tier_At_Risk personalized campaign\n')
strategy_lines.append('5. **Week 3-4:** Win-back sequence for Dormant customers\n')
strategy_lines.append('6. **Ongoing:** VIP program for Champions, onboarding for New_Customers\n')

with open('retention_strategy.md', 'w') as f:
    f.writelines(strategy_lines)

print('✓ Saved retention_strategy.md')

## 8. Manual Review Cases

In [ ]:
# Flag ambiguous customers: mixed high/low signals
snap['conflict_score'] = 0
snap.loc[(snap['monetary_180d'] > snap['monetary_180d'].quantile(0.75)) &
         (snap['ticket_count_90d'] >= 2), 'conflict_score'] += 2
snap.loc[(snap['avg_discount_pct_180d'] > 0.4) &
         (snap['recency_days'].between(31, 90)), 'conflict_score'] += 2
snap.loc[(snap['sessions_30d'] >= 5) &
         (snap['recency_days'] > 60), 'conflict_score'] += 2
snap.loc[(snap['return_rate_180d'] > 0.3) &
         (snap['frequency_180d'] >= 3), 'conflict_score'] += 1

review = snap.nlargest(15, 'conflict_score')[
    ['customer_id', 'segment_name', 'recency_days', 'frequency_180d',
     'monetary_180d', 'ticket_count_90d', 'return_rate_180d',
     'avg_discount_pct_180d', 'sessions_30d', 'churn_next_60d', 'conflict_score']
]
review

## 9. Save Manual Review Cases Markdown

In [ ]:
# Generate and save manual review cases markdown
review_lines = ['# Manual Review Cases\n']
review_lines.append(f'Generated: {pd.Timestamp.now().strftime("%Y-%m-%d")}\n')
review_lines.append(f'Snapshot date: 2025-09-30\n\n')

review_lines.append('## Overview\n\n')
review_lines.append('These 15 customers have conflicting signals (mixed high/low engagement/value) and merit manual review ')
review_lines.append('for segment reassignment or targeted intervention.\n\n')

review_lines.append('**Conflict Score Legend:**\n')
review_lines.append('- **+2 points:** High spend (>75th percentile) + 2+ support tickets (unhappy high-value)\n')
review_lines.append('- **+2 points:** High discount usage (>40%) + moderate recency 31-90d (lapsed but price-sensitive)\n')
review_lines.append('- **+2 points:** High engagement (≥5 sessions) + old recency >60d (active but long time since purchase)\n')
review_lines.append('- **+1 point:** High returns (>30%) + frequent buyer (quality concern)\n\n')

review_lines.append('## Top 15 Manual Review Cases\n\n')
review_lines.append('| Customer ID | Segment | Recency | Frequency | Spend (₹) | Tickets | Return % | Discount % | Sessions | Churn | Score |\n')
review_lines.append('|---|---|---:|---:|---:|---:|---:|---:|---:|:---:|---:|\n')

for idx, row in review.iterrows():
    review_lines.append(
        f"| {row['customer_id']} | {row['segment_name']} | {row['recency_days']:.0f}d | {row['frequency_180d']:.0f} | "
        f"₹{row['monetary_180d']:.0f} | {row['ticket_count_90d']:.0f} | {row['return_rate_180d']:.0%} | "
        f"{row['avg_discount_pct_180d']:.0%} | {row['sessions_30d']:.0f} | {row['churn_next_60d']:.0f} | {row['conflict_score']:.0f} |\n"
    )

review_lines.append('\n## Recommended Actions by Conflict Type\n\n')
review_lines.append('### Type 1: High-Value + Support Issues (upgrade to "Support Priority")\n')
review_lines.append('Customers with high spend but 2+ support tickets.\n')
review_lines.append('**Action:** Pro-active support outreach; offer loyalty credit; assess satisfaction.\n')
review_lines.append('**Expected Outcome:** Convert to Champions or stabilize in High_Value_Unhappy.\n\n')

review_lines.append('### Type 2: Price-Sensitive with Lapsed Recency (reactivation offer)\n')
review_lines.append('Customers with high discount usage but 31-90 days since last purchase.\n')
review_lines.append('**Action:** Limited-time bundle or milestone offer; re-engagement campaign.\n')
review_lines.append('**Expected Outcome:** Move to Loyal_Slipping or back to active buying.\n\n')

review_lines.append('### Type 3: Engaged but Long Recency (conversion friction?)\n')
review_lines.append('Customers with 5+ recent sessions but >60 days since purchase.\n')
review_lines.append('**Action:** Exit-intent survey; check cart abandonment; product rec email.\n')
review_lines.append('**Expected Outcome:** Understand conversion barrier; improve offer relevance.\n\n')

review_lines.append('### Type 4: Quality Concern (high returns + repeat buyer)\n')
review_lines.append('Customers with >30% return rate but multiple repeat purchases.\n')
review_lines.append('**Action:** Quality/fit issue investigation; product education; size guide/review.\n')
review_lines.append('**Expected Outcome:** Reduce returns; improve CLTV; lower churn.\n\n')

with open('manual_review_cases.md', 'w') as f:
    f.writelines(review_lines)

print('✓ Saved manual_review_cases.md')